# jupyter_cache

jupyter-cache 1.0.1 keeps executed notebooks in a cache keyed by a hash of their code cells (SQLite through SQLAlchemy),
so an unchanged notebook is not executed again; MyST-NB's "cache" execution mode is built on it. These cells add a
notebook to a cache project, execute it, read the outputs back, show that only the code changes the hash, and use the
`jcache` command.

In [ ]:
# Example 1: a cache, a project notebook and the local executor: the notebook is run once and cached.
import pathlib, tempfile
import nbformat
from jupyter_cache import get_cache
from jupyter_cache.executors import load_executor

work = pathlib.Path(tempfile.mkdtemp())
nb = nbformat.v4.new_notebook(cells=[nbformat.v4.new_code_cell("print(6 * 7)")])
nb.metadata["kernelspec"] = {"name": "python3", "display_name": "Python 3", "language": "python"}
nbformat.write(nb, work / "a.ipynb")
cache = get_cache(work / ".cache")
cache.add_nb_to_project(str(work / "a.ipynb"))
result = load_executor("local-serial", cache).run_and_cache()
print(result)
assert result.succeeded == [str(work / "a.ipynb")] and len(cache.list_cache_records()) == 1

In [ ]:
# Example 2: the cached outputs are merged back into the source notebook without executing it again.
import pathlib, tempfile
import nbformat
from jupyter_cache import get_cache
from jupyter_cache.executors import load_executor

work = pathlib.Path(tempfile.mkdtemp())
nb = nbformat.v4.new_notebook(cells=[nbformat.v4.new_code_cell("print('cached-out')")])
nb.metadata["kernelspec"] = {"name": "python3", "display_name": "Python 3", "language": "python"}
nbformat.write(nb, work / "b.ipynb")
cache = get_cache(work / ".cache")
cache.add_nb_to_project(str(work / "b.ipynb"))
load_executor("local-serial", cache).run_and_cache()
_, merged = cache.merge_match_into_file(str(work / "b.ipynb"))
print(merged.cells[0].outputs)
assert merged.cells[0].outputs[0]["text"] == "cached-out\n"

In [ ]:
# Example 3: the cache key is the code: a markdown edit keeps the hash, a code edit changes it.
import tempfile
import nbformat
from jupyter_cache import get_cache

def nb(code, md):
    return nbformat.v4.new_notebook(cells=[nbformat.v4.new_markdown_cell(md), nbformat.v4.new_code_cell(code)])

cache = get_cache(tempfile.mkdtemp())
h1 = cache.create_hashed_notebook(nb("x = 1", "first words"))[1]
h2 = cache.create_hashed_notebook(nb("x = 1", "other words"))[1]
h3 = cache.create_hashed_notebook(nb("x = 2", "first words"))[1]
print(h1, h3)
assert h1 == h2 and h1 != h3

In [ ]:
# Example 4: a notebook that raises is recorded as failed, with its traceback kept for the report.
import pathlib, tempfile
import nbformat
from jupyter_cache import get_cache
from jupyter_cache.executors import load_executor

work = pathlib.Path(tempfile.mkdtemp())
nb = nbformat.v4.new_notebook(cells=[nbformat.v4.new_code_cell("raise ValueError('boom')")])
nb.metadata["kernelspec"] = {"name": "python3", "display_name": "Python 3", "language": "python"}
nbformat.write(nb, work / "bad.ipynb")
cache = get_cache(work / ".cache")
record = cache.add_nb_to_project(str(work / "bad.ipynb"))
result = load_executor("local-serial", cache).run_and_cache()
print(result)
assert result.excepted == [str(work / "bad.ipynb")] and result.succeeded == []
assert "boom" in cache.get_project_record(record.pk).traceback

In [ ]:
# Example 5: the jcache command lists a project's notebooks and the cache's records.
import pathlib, subprocess, tempfile
import nbformat

work = pathlib.Path(tempfile.mkdtemp())
nbformat.write(nbformat.v4.new_notebook(cells=[nbformat.v4.new_code_cell("1")]), work / "c.ipynb")
cache_path = str(work / ".cache")
add = subprocess.run(["jcache", "notebook", "-p", cache_path, "add", str(work / "c.ipynb")], input="y\n",
                     capture_output=True, text=True)
listing = subprocess.run(["jcache", "notebook", "-p", cache_path, "list"], capture_output=True, text=True)
print(add.returncode, listing.returncode, listing.stdout)
assert add.returncode == 0 and listing.returncode == 0
assert "c.ipynb" in listing.stdout